# 5장 3강: 배깅 vs 부스팅 선택과 모델 해석(SHAP)

## 실습 목표

이번 실습에서는 **Ames Housing** 데이터를 이용하여 배깅과 부스팅을 비교하고, SHAP으로 앙상블 모델의 예측 근거를 해석합니다.

- Random Forest와 XGBoost의 구조적 차이를 비교합니다.
- 성능과 학습 시간을 확인하여 상황에 맞는 모델 선택 기준을 정리합니다.
- Feature Importance의 특징과 한계를 확인합니다.
- TreeSHAP을 이용해 모델 전체의 **전역 해석**을 수행합니다.
- 특정 주택 한 건의 예측을 **지역 해석**합니다.
- SHAP 값의 부호를 이용하여 각 특성이 예측값을 올렸는지 내렸는지 설명합니다.

> 이 실습에서는 5장 3강 교안에 나온 배깅/부스팅 비교, Feature Importance, SHAP 전역·지역 해석 내용만 사용합니다.

## 실습 환경 / 데이터

- Python
- pandas
- matplotlib
- scikit-learn
- xgboost
- shap
- 데이터: `data/Ames_Housing.csv`

### 분석 목표

주택 특성을 이용하여 `SalePrice`를 예측하는 회귀 문제입니다.

### 사용할 특성

- `OverallQual`
- `GrLivArea`
- `GarageCars`
- `GarageArea`
- `TotalBsmtSF`
- `1stFlrSF`
- `2ndFlrSF`
- `FullBath`
- `TotRmsAbvGrd`
- `YearBuilt`
- `YearRemodAdd`
- `Fireplaces`

예측 대상은 `SalePrice`입니다.

## 실습 준비

1. Ames Housing 데이터를 불러옵니다.
2. 사용할 숫자형 특성과 `SalePrice`를 선택합니다.
3. Train / Test 데이터로 분리합니다.
4. Random Forest와 XGBoost를 같은 데이터에서 비교합니다.
5. SHAP은 XGBoost 모델을 대상으로 확인합니다.

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

house = pd.read_csv("data/Ames_Housing.csv")

features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "GarageArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "FullBath",
    "TotRmsAbvGrd",
    "YearBuilt",
    "YearRemodAdd",
    "Fireplaces"
]

X = house[features]
y = house["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train 크기:", X_train.shape)
print("Test 크기:", X_test.shape)

print("\n[결측치]")
display(X.isna().sum().to_frame("missing"))

---

# 필수 1. 배깅과 부스팅 비교 및 선택

## 문제 1. Random Forest와 XGBoost를 비교하세요.

### 요구사항

다음 두 모델을 사용합니다.

```python
RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)
```

각 모델에 대해 다음을 확인합니다.

1. Train 데이터로 학습합니다.
2. 학습 시간을 측정합니다.
3. Test 데이터를 예측합니다.
4. Test RMSE를 계산합니다.
5. Test R²를 계산합니다.
6. 결과를 하나의 DataFrame으로 정리합니다.

### 확인 포인트

- Random Forest가 배깅 계열이라는 점을 이해했는가?
- XGBoost가 부스팅 계열이라는 점을 이해했는가?
- 성능뿐 아니라 학습 시간도 함께 확인했는가?

In [ ]:
# TODO
# Random Forest와 XGBoost의 Test RMSE, Test R², 학습 시간을 비교하세요.
rf = RandomForestRegressor(n_estimators=200, random_state=42)
t0 = time.time()
rf.fit(X_train, y_train)
rf_time = time.time() - t0
rf_pred = rf.predict(X_test)

xgb = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42)
t0 = time.time()
xgb.fit(X_train, y_train)
xgb_time = time.time() - t0
xgb_pred = xgb.predict(X_test)

compare_df = pd.DataFrame([
    {
        "model": "RandomForest",
        "fit_seconds": rf_time,
        "test_rmse": mean_squared_error(y_test, rf_pred) ** 0.5,
        "test_r2": r2_score(y_test, rf_pred),
    },
    {
        "model": "XGBoost",
        "fit_seconds": xgb_time,
        "test_rmse": mean_squared_error(y_test, xgb_pred) ** 0.5,
        "test_r2": r2_score(y_test, xgb_pred),
    },
])
display(compare_df)


### Q1. Random Forest와 XGBoost의 학습 방식은 어떻게 다른가요?

**답변:**  
Random Forest는 배깅(병렬·평균)이고, XGBoost는 부스팅(순차적으로 잔차 보완)입니다.


### Q2. 다음 상황에서 어떤 방식을 우선 후보로 생각할 수 있는지 작성하세요.

1. 빠르게 베이스라인을 만들고 싶고 튜닝 여력이 적음 → Random Forest(배깅)
2. 시간을 들여 예측 성능을 더 끌어올리고 싶음 → XGBoost(부스팅)
3. 노이즈가 많아 과적합 위험을 조심해야 함 → Random Forest를 우선 후보로 검토

**답변:**  
위와 같습니다.


---

# 필수 2. Feature Importance와 SHAP 전역 해석

## 문제 1. XGBoost의 Feature Importance를 확인하세요.

### 요구사항

1. 필수 1에서 학습한 XGBoost 모델을 사용합니다.
2. `feature_importances_`를 이용합니다.
3. 특성명과 중요도를 DataFrame으로 만듭니다.
4. 중요도가 높은 순서대로 정렬합니다.
5. 막대그래프로 표시합니다.

### 확인 포인트

- 어떤 특성이 모델 전체에서 중요하게 사용되었는지 확인했는가?
- 중요도의 크기는 알 수 있지만 방향은 알 수 없다는 점을 이해했는가?

In [ ]:
# TODO
# XGBoost의 Feature Importance를 표와 그래프로 확인하세요.
fi = pd.DataFrame({
    "feature": X.columns,
    "importance": xgb.feature_importances_,
}).sort_values("importance", ascending=False)
display(fi)

plt.figure(figsize=(8, 5))
plt.barh(fi["feature"][::-1], fi["importance"][::-1])
plt.title("XGBoost Feature Importance")
plt.tight_layout()
plt.show()


### Q3. Feature Importance만으로는 알기 어려운 정보는 무엇인가요?

**답변:**  
특성이 예측을 올렸는지/내렸는지(방향)와 개별 샘플에서의 기여도를 알기 어렵습니다.


## 문제 2. TreeSHAP으로 전역 중요도를 확인하세요.

### 요구사항

1. `shap.TreeExplainer()`에 XGBoost 모델을 전달합니다.
2. Test 데이터의 SHAP 값을 계산합니다.
3. `shap.plots.bar()`로 전역 중요도를 확인합니다.
4. SHAP 절댓값의 평균도 계산하여 특성별 중요도를 DataFrame으로 정리합니다.
5. 상위 특성을 확인합니다.

### 확인 포인트

- SHAP은 각 특성이 예측값에 기여한 정도를 계산한다는 점을 이해했는가?
- 전역 SHAP에서는 여러 샘플의 기여도 크기를 종합해 전체 중요도를 볼 수 있는가?

In [ ]:
# TODO
# TreeSHAP을 이용해 XGBoost 모델의 전역 SHAP 중요도를 확인하세요.
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)
shap.plots.bar(shap.Explanation(values=shap_values, data=X_test.values, feature_names=list(X.columns)), show=True)
# 호환을 위한 대체 호출
try:
    shap.summary_plot(shap_values, X_test, plot_type="bar", show=True)
except Exception:
    pass


### Q4. Feature Importance와 SHAP의 가장 큰 차이는 무엇인가요?

**답변:**  
Feature Importance는 전역 사용 빈도/이득 중심이고, SHAP은 기여의 방향(+/-)과 개별 예측 설명까지 제공합니다.


---
# 과제 1. 모델 비교·SHAP 해석과 특성 선택 전후 검증

## 배경
주택 가격 예측 모델을 성능과 학습 시간으로 비교하고, SHAP으로 예측 근거를 설명합니다. 이어서 중요한 특성을 선택했을 때 검증 성능이 어떻게 달라지는지 확인합니다.

> 필수 1·2는 연습용입니다. 아래 과제의 비교표·해석·코드를 제출하세요. 준비 단계의 `X_train`, `y_train`을 사용하되 과제용 모델을 새로 학습합니다. Test 결과로 모델이나 특성을 선택하지 않도록, Train 안에서 Validation을 분리합니다. 개별 주택 해석 대상도 Validation의 첫 번째 주택으로 맞춥니다.

## 요구사항

### 1. 같은 조건에서 모델 비교
1. `X_train`, `y_train`을 Train_sub / Validation으로 나누세요(`test_size=0.2`, `random_state=42`). 결측값이 있으면 Train_sub에서 구한 중앙값으로 두 데이터를 채우세요.
2. Random Forest(`n_estimators=200`)와 XGBoost(`n_estimators=200`, `learning_rate=0.05`, `max_depth=3`)를 같은 Train_sub에서 학습하세요. 두 모델 모두 `random_state=42`, `n_jobs=1`로 설정하세요.
3. 각 모델의 **fit에 걸린 시간**, Train_sub RMSE, Validation RMSE와 R²를 `model_comparison`으로 정리하세요. 시간은 같은 환경에서 측정하며, 한 번의 측정만으로 일반적인 속도 우열을 단정하지 마세요.
4. 예측 오차, 학습 시간, Train/Validation 오차 차이와 운영 목적을 근거로 우선 후보를 선택하세요. 모델 선택 결과와 관계없이 다음 해석은 XGBoost로 수행합니다.

### 2. Feature Importance와 SHAP 전역·지역 해석
5. 과제용 XGBoost의 `feature_importances_`와 **Train_sub의 평균 절댓값 SHAP**을 각각 순위로 정리하여 `importance_comparison`으로 비교하세요. 전역 SHAP 막대그래프도 제출하세요.
6. Validation의 첫 번째 주택의 실제 가격·예측가격을 확인하고 SHAP waterfall plot을 그리세요.
7. 해당 주택의 절댓값 SHAP 상위 5개 특성·특성값·SHAP 값을 `local_top5`로 정리하세요. 양수·음수의 의미를 설명하고 기준값 + 모든 SHAP 값의 합이 예측값과 거의 같은지 확인하세요.
8. 두 전역 중요도 산출 원리와 한계를 비교하고, 전역 순위와 개별 주택의 기여도가 다른 이유를 설명하세요.

### 3. 중요도에 근거한 특성 선택 및 검증
9. Train_sub의 평균 절댓값 SHAP이 높은 **상위 6개 특성**을 선택하세요. 특성 선택은 이번 과제의 특성 개선 시도입니다. 개수는 비교 결과를 보기 전에 6개로 고정합니다.
10. 선택한 특성만 사용하여 동일한 XGBoost 설정으로 다시 학습하세요. 기존 모델과 동일한 Train_sub / Validation 행을 사용하고, 특성 이외의 조건은 유지하세요.
11. 전체 특성 모델과 선택 특성 모델의 특성 수·학습 시간·Train RMSE·Validation RMSE·Validation R²를 `feature_comparison`으로 비교하세요. `기존 Validation RMSE - 선택 후 Validation RMSE`와 기존 대비 개선율(%)을 계산하세요.
12. 개선 여부와 채택 여부를 수치로 설명하세요. 성능이 나빠져도 원인 가능성과 한계를 타당하게 설명하면 됩니다. 검증 결과를 반복 확인하며 특성을 바꾸지 말고, 최종 일반화 성능 확인에는 별도 Test 평가가 필요함을 밝히세요.

## 해석 질문

### Q5. 전역 SHAP에서 가장 중요한 특성이 특정 주택에서도 반드시 가장 큰 영향을 주나요?
### Q6. SHAP 값이 양수/음수라는 것은 각각 무엇을 의미하나요?
### Q7. 두 모델 중 무엇을 우선 후보로 선택했나요? 성능·학습 시간·과적합 징후와 주택 가격 예측 업무를 연결하여 설명하세요.
### Q8. Feature Importance와 평균 절댓값 SHAP의 계산 원리와 한계는 무엇이며, 실제 상위 특성 순위는 어떻게 다른가요?
### Q9. 상위 6개 특성은 무엇이며, 선택 전후 RMSE 차이와 개선율은 얼마인가요? 이 시도를 채택할지와 판단의 한계를 설명하세요.

## 제출물 / 확인 포인트
- `model_comparison`, `importance_comparison`, 전역 SHAP 그래프
- 개별 주택의 실제값·예측값·waterfall plot·`local_top5`·SHAP 합 검증
- 선택한 6개 특성, `feature_comparison`, RMSE 차이와 개선율
- 계산 코드와 Q5~Q9 답변
- 동일한 검증 조건 유지, SHAP의 방향·전역/지역 구분, 관측한 개선 여부와 해석의 한계

In [ ]:
# TODO 1: 같은 Train_sub / Validation으로 두 모델의 성능과 fit 시간을 비교하세요.
X_tr_sub, X_val, y_tr_sub, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# 결측 중앙값 대체(있을 경우)
medians = X_tr_sub.median(numeric_only=True)
X_tr_sub = X_tr_sub.fillna(medians)
X_val = X_val.fillna(medians)

rows = []
rf_task = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=1)
t0 = time.time()
rf_task.fit(X_tr_sub, y_tr_sub)
rf_fit = time.time() - t0
rf_tr_pred = rf_task.predict(X_tr_sub)
rf_va_pred = rf_task.predict(X_val)
rows.append({
    "model": "RandomForest",
    "fit_seconds": rf_fit,
    "train_rmse": mean_squared_error(y_tr_sub, rf_tr_pred) ** 0.5,
    "valid_rmse": mean_squared_error(y_val, rf_va_pred) ** 0.5,
    "valid_r2": r2_score(y_val, rf_va_pred),
})

xgb_task = XGBRegressor(
    n_estimators=200, learning_rate=0.05, max_depth=3,
    random_state=42, n_jobs=1,
)
t0 = time.time()
xgb_task.fit(X_tr_sub, y_tr_sub)
xgb_fit = time.time() - t0
xgb_tr_pred = xgb_task.predict(X_tr_sub)
xgb_va_pred = xgb_task.predict(X_val)
rows.append({
    "model": "XGBoost",
    "fit_seconds": xgb_fit,
    "train_rmse": mean_squared_error(y_tr_sub, xgb_tr_pred) ** 0.5,
    "valid_rmse": mean_squared_error(y_val, xgb_va_pred) ** 0.5,
    "valid_r2": r2_score(y_val, xgb_va_pred),
})

model_comparison = pd.DataFrame(rows)
display(model_comparison)


In [ ]:
# TODO 2: 과제용 XGBoost의 Feature Importance, Train_sub 전역 SHAP, Validation 첫 주택의 지역 SHAP을 확인하세요.
fi_task = pd.DataFrame({
    "feature": X_tr_sub.columns,
    "fi": xgb_task.feature_importances_,
})

explainer_task = shap.TreeExplainer(xgb_task)
shap_train = explainer_task.shap_values(X_tr_sub)
mean_abs_shap = np.abs(shap_train).mean(axis=0)
shap_rank = pd.DataFrame({
    "feature": X_tr_sub.columns,
    "mean_abs_shap": mean_abs_shap,
})

importance_comparison = fi_task.merge(shap_rank, on="feature")
importance_comparison["fi_rank"] = importance_comparison["fi"].rank(ascending=False)
importance_comparison["shap_rank"] = importance_comparison["mean_abs_shap"].rank(ascending=False)
importance_comparison = importance_comparison.sort_values("shap_rank")
display(importance_comparison)

shap.summary_plot(shap_train, X_tr_sub, plot_type="bar", show=True)

# Validation 첫 주택 지역 해석
x0 = X_val.iloc[[0]]
y0_true = y_val.iloc[0]
y0_pred = xgb_task.predict(x0)[0]
print("실제 가격:", y0_true)
print("예측 가격:", y0_pred)

shap_val0 = explainer_task.shap_values(x0)[0]
base_value = explainer_task.expected_value
if isinstance(base_value, (list, np.ndarray)):
    base_value = float(np.array(base_value).ravel()[0])

local_df = pd.DataFrame({
    "feature": X_val.columns,
    "feature_value": x0.values.ravel(),
    "shap": shap_val0,
})
local_df["abs_shap"] = local_df["shap"].abs()
local_top5 = local_df.sort_values("abs_shap", ascending=False).head(5)
display(local_top5)

print("base + sum(shap) ≈", base_value + shap_val0.sum(), " / pred=", y0_pred)

try:
    shap.plots.waterfall(
        shap.Explanation(
            values=shap_val0,
            base_values=base_value,
            data=x0.values.ravel(),
            feature_names=list(X_val.columns),
        )
    )
except Exception as e:
    print("waterfall plot skip:", e)


In [ ]:
# TODO 3: Train_sub SHAP 상위 6개 특성으로 재학습하고 동일 Validation에서 전후 성능을 비교하세요.
top6 = importance_comparison.sort_values("mean_abs_shap", ascending=False)["feature"].head(6).tolist()
print("상위 6개 특성:", top6)

xgb_full = xgb_task  # 기존 전체 특성 모델
full_valid_rmse = mean_squared_error(y_val, xgb_full.predict(X_val)) ** 0.5
full_valid_r2 = r2_score(y_val, xgb_full.predict(X_val))
full_train_rmse = mean_squared_error(y_tr_sub, xgb_full.predict(X_tr_sub)) ** 0.5

xgb_sel = XGBRegressor(
    n_estimators=200, learning_rate=0.05, max_depth=3,
    random_state=42, n_jobs=1,
)
t0 = time.time()
xgb_sel.fit(X_tr_sub[top6], y_tr_sub)
sel_fit = time.time() - t0
sel_train_rmse = mean_squared_error(y_tr_sub, xgb_sel.predict(X_tr_sub[top6])) ** 0.5
sel_valid_rmse = mean_squared_error(y_val, xgb_sel.predict(X_val[top6])) ** 0.5
sel_valid_r2 = r2_score(y_val, xgb_sel.predict(X_val[top6]))

rmse_diff = full_valid_rmse - sel_valid_rmse
improve_pct = (rmse_diff / full_valid_rmse) * 100 if full_valid_rmse != 0 else np.nan

feature_comparison = pd.DataFrame([
    {
        "setting": "all_features",
        "n_features": X_tr_sub.shape[1],
        "fit_seconds": model_comparison.loc[model_comparison["model"]=="XGBoost", "fit_seconds"].values[0],
        "train_rmse": full_train_rmse,
        "valid_rmse": full_valid_rmse,
        "valid_r2": full_valid_r2,
    },
    {
        "setting": "top6_shap",
        "n_features": len(top6),
        "fit_seconds": sel_fit,
        "train_rmse": sel_train_rmse,
        "valid_rmse": sel_valid_rmse,
        "valid_r2": sel_valid_r2,
    },
])
display(feature_comparison)
print("RMSE diff (full - sel):", rmse_diff)
print("개선율(%):", improve_pct)


## Q5~Q9 답변

- Q5: 아닙니다. 전역 중요도가 높은 특성이 특정 주택에서는 기여가 작을 수 있습니다(지역 SHAP이 다름).
- Q6: 양수는 예측값을 기준(기대값)보다 올리고, 음수는 내리는 기여를 의미합니다.
- Q7: `model_comparison`의 Validation RMSE·학습 시간·Train/Valid 간격을 보고 우선 후보를 고릅니다. 해석은 XGBoost로 수행했습니다.
- Q8: FI는 분할 이득/빈도 기반이라 방향이 없고 편향될 수 있고, 평균 |SHAP|은 기여 크기의 평균입니다. 순위는 `importance_comparison`에서 비교합니다.
- Q9: 상위 6개와 RMSE 차이·개선율은 위 코드 출력을 근거로 채택 여부를 판단합니다. Validation만으로 확정하지 말고 별도 Test가 필요합니다.


---

# 실습 마무리

아래 질문에 짧게 답해보세요.

1. 배깅과 부스팅의 학습 방식 차이는 무엇인가요?
2. 배깅과 부스팅은 각각 편향과 분산 중 무엇을 줄이는 데 초점을 두나요?
3. 튜닝 여력이 부족한 상황에서는 어떤 모델군을 먼저 고려할 수 있나요?
4. Feature Importance의 가장 큰 한계는 무엇인가요?
5. SHAP의 전역 해석과 지역 해석은 무엇이 다른가요?
6. SHAP 값의 양수와 음수는 무엇을 의미하나요?